# Stage 0 — Signal Validity Test

Tests whether shape-conditioned trading signals predict anything at all.
Does NOT test profitability — only whether Signal ON days behave differently from Signal OFF days.

- **Strategy A**: Mean reversion in SB (0.0) and C (1) — 9 instruments (5 calendar pairs + 4 butterflies)
- **Strategy B**: Trend following in MB (0.1), TB (0.2), F (2) — 5 calendar pairs

All model logic imported from `models/` — no inline reimplementation.

In [1]:
import sys
sys.path.insert(0, r'C:/ClaudeCode')

import pandas as pd
import numpy as np
from datetime import datetime
from models.pm_engine import predict as pm_predict
from models.tm_engine import predict as tm_predict
from models.feature_prep import load_daily_shape_log, load_enriched_shape_log, TM_FEATURES

LOG_FILE = r'C:/ClaudeCode/research/04. backtest_analysis/backtest_analysis.txt'

# Train/test split
OOS_START = '2022-01-01'  # Out-of-sample starts here
# In-sample: everything before OOS_START
# Model features available from 2017+ (enriched shape log starts 2017)
# Shape log with M1-M6 available from 2008+

print('Imports OK')

Imports OK


In [2]:
# ══════════════════════════════════════════════════════════════
# CELL 2 — DATA SETUP
# ══════════════════════════════════════════════════════════════

# Load full shape log (2008+) for M1-M6 prices and shape history
full_log = load_daily_shape_log()
full_log = full_log.sort_values('date').reset_index(drop=True)

# Load enriched log (2017+) for days_in_shape, duration_stage, prior_shape
enriched = load_enriched_shape_log()
enriched = enriched.sort_values('date').reset_index(drop=True)

# For 2008-2016 data, compute days_in_shape and duration_stage from full_log
pre_2017 = full_log[full_log['date'] < '2017-01-01'].copy()

# Compute episode tracking for pre-2017
pre_2017 = pre_2017.sort_values('date').reset_index(drop=True)
days_list = []
prior_list = []
episode_list = []
ep_id = 0
prev_shape = None
day_count = 0
for i, row in pre_2017.iterrows():
    if row['shape'] != prev_shape:
        prior_list.append(prev_shape)
        prev_shape = row['shape']
        day_count = 1
        ep_id += 1
    else:
        prior_list.append(prev_shape if day_count == 1 else prior_list[-1] if prior_list else None)
        day_count += 1
    days_list.append(day_count)
    episode_list.append(ep_id)

# Fix prior_shape: it should be the shape BEFORE the current episode
prior_shapes_fixed = []
prev_ep_shape = None
current_ep = None
for i, row in pre_2017.iterrows():
    if episode_list[i] != current_ep:
        prior_shapes_fixed.append(prev_ep_shape)
        prev_ep_shape = pre_2017.iloc[i-1]['shape'] if i > 0 else None
        current_ep = episode_list[i]
    else:
        prior_shapes_fixed.append(prior_shapes_fixed[-1])

pre_2017['days_in_shape'] = days_list
pre_2017['prior_shape'] = prior_shapes_fixed
pre_2017['episode_id'] = episode_list
pre_2017['duration_stage'] = pre_2017['days_in_shape'].apply(
    lambda d: 'EARLY' if d <= 7 else ('SETTLING' if d <= 19 else 'ESTABLISHED'))

# Combine: use enriched for 2017+, pre_2017 for earlier
shared_cols = ['date', 'shape', 'prior_shape', 'days_in_shape', 'duration_stage',
               'M1', 'M2', 'M3', 'M4', 'M5', 'M6']

# Add episode_id to enriched if not present
if 'episode_id' not in enriched.columns:
    enriched['episode_id'] = (enriched['shape'] != enriched['shape'].shift(1)).cumsum() + ep_id

shared_cols_with_ep = shared_cols + ['episode_id']
df = pd.concat([
    pre_2017[shared_cols_with_ep],
    enriched[shared_cols_with_ep]
], ignore_index=True).sort_values('date').reset_index(drop=True)

# Verify no duplicates
df = df.drop_duplicates(subset='date', keep='last').reset_index(drop=True)

print(f'Combined daily panel: {len(df)} rows')
print(f'Date range: {df["date"].min().date()} to {df["date"].max().date()}')
print(f'Shape distribution:\n{df["shape"].value_counts().sort_index()}')
print(f'\nDuration stage distribution:\n{df["duration_stage"].value_counts()}')

Combined daily panel: 4070 rows
Date range: 2008-01-01 to 2026-06-26
Shape distribution:
shape
0.0    1330
0.1     588
0.2     372
1      1219
2       561
Name: count, dtype: int64

Duration stage distribution:
duration_stage
EARLY          2159
ESTABLISHED     983
SETTLING        878
LATE             50
Name: count, dtype: int64


In [3]:
# ══════════════════════════════════════════════════════════════
# CELL 3 — SPREAD & BUTTERFLY COMPUTATION + Z-SCORES
# ══════════════════════════════════════════════════════════════

# Define instruments
CALENDAR_PAIRS = [
    ('M1-M2', 'M1', 'M2'),
    ('M2-M3', 'M2', 'M3'),
    ('M3-M4', 'M3', 'M4'),
    ('M4-M5', 'M4', 'M5'),
    ('M5-M6', 'M5', 'M6'),
]

BUTTERFLIES = [
    ('BF_M1M2M3', 'M1', 'M2', 'M3'),
    ('BF_M2M3M4', 'M2', 'M3', 'M4'),
    ('BF_M3M4M5', 'M3', 'M4', 'M5'),
    ('BF_M4M5M6', 'M4', 'M5', 'M6'),
]

# Compute spread values
for name, near, far in CALENDAR_PAIRS:
    df[name] = df[near] - df[far]

# Compute butterfly values
for name, m1, m2, m3 in BUTTERFLIES:
    df[name] = df[m1] - 2 * df[m2] + df[m3]

ALL_INSTRUMENTS = [p[0] for p in CALENDAR_PAIRS] + [b[0] for b in BUTTERFLIES]
STRAT_A_INSTRUMENTS = ALL_INSTRUMENTS  # 5 calendar + 4 butterfly
STRAT_B_INSTRUMENTS = [p[0] for p in CALENDAR_PAIRS]  # 5 calendar only

# ── Regime-relative z-score computation ─────────────────────
# Z-score resets when shape changes, uses rolling window within current shape episode
# Min 10 days before z-score is valid; window = min(60, episode length)

def compute_regime_zscore(df, instrument_col):
    """Compute regime-relative z-score for one instrument.
    Returns Series aligned with df index."""
    zscores = pd.Series(np.nan, index=df.index)

    for ep_id in df['episode_id'].unique():
        mask = df['episode_id'] == ep_id
        ep_vals = df.loc[mask, instrument_col]

        if len(ep_vals) < 10:
            continue  # z-score not valid with < 10 days

        # Rolling within episode: window = min(60, episode length so far)
        for i, (idx, val) in enumerate(ep_vals.items()):
            if i < 9:  # need at least 10 days (index 0-9)
                continue
            window_start = max(0, i - 59)  # 60-day max window
            window = ep_vals.iloc[window_start:i+1]
            mean = window.mean()
            std = window.std()
            if std > 0 and not np.isnan(val):
                zscores[idx] = (val - mean) / std

    return zscores

# Compute z-scores for all instruments
print('Computing regime-relative z-scores...')
for inst in ALL_INSTRUMENTS:
    df[f'{inst}_z'] = compute_regime_zscore(df, inst)
    valid_n = df[f'{inst}_z'].notna().sum()
    print(f'  {inst}: {valid_n} valid z-score days')

print('\nZ-score computation complete.')

Computing regime-relative z-scores...


  M1-M2: 1975 valid z-score days


  M2-M3: 1975 valid z-score days


  M3-M4: 1975 valid z-score days


  M4-M5: 1974 valid z-score days


  M5-M6: 1970 valid z-score days
  BF_M1M2M3: 1975 valid z-score days


  BF_M2M3M4: 1975 valid z-score days


  BF_M3M4M5: 1975 valid z-score days


  BF_M4M5M6: 1975 valid z-score days

Z-score computation complete.


In [4]:
# ══════════════════════════════════════════════════════════════
# CELL 4 — MODEL OUTPUTS (PM + TM, computed once per day)
# ══════════════════════════════════════════════════════════════
# Models only available for dates where TM features exist (2017+).
# For 2008-2016, PM/TM outputs are NaN — signal conditions requiring
# model outputs will only fire on 2017+ data.

# PM confidence threshold for Level 0 vs Level 1
# From calibration: 90%+ bucket has 93.5% accuracy, 70-90% has 78.9%
# Use 70% as the Level 0 threshold (PM confirms with high confidence)
PM_CONFIDENCE_THRESHOLD = 0.70

# TM 1w cell ratings from latest calibration (post XGB swap, 2026-07-10 13:06)
TM_1W_RATINGS = {
    ('0.0', '0.0'): 'IGNORE', ('0.0', '0.1'): 'IGNORE', ('0.0', '0.2'): 'USE',
    ('0.0', '1'): 'USE', ('0.0', '2'): 'SOFT',
    ('0.1', '0.0'): 'USE', ('0.1', '0.1'): 'IGNORE', ('0.1', '0.2'): 'IGNORE',
    ('0.1', '1'): 'USE', ('0.1', '2'): 'IGNORE',
    ('0.2', '0.0'): 'SKIP', ('0.2', '0.1'): 'SKIP', ('0.2', '0.2'): 'SKIP',
    ('0.2', '1'): 'SKIP', ('0.2', '2'): 'SKIP',
    ('1', '0.0'): 'IGNORE', ('1', '0.1'): 'IGNORE', ('1', '0.2'): 'IGNORE',
    ('1', '1'): 'IGNORE', ('1', '2'): 'IGNORE',
    ('2', '0.0'): 'IGNORE', ('2', '0.1'): 'IGNORE', ('2', '0.2'): 'IGNORE',
    ('2', '1'): 'USE', ('2', '2'): 'USE',
}

# Run model predictions for dates in 2017+ range
model_start = pd.Timestamp('2017-01-01')
model_dates = df[df['date'] >= model_start].copy()

print(f'Running PM + TM predictions for {len(model_dates)} dates (2017+)...')

pm_results = []
tm_1w_results = []
tm_2w_results = []

for i, (_, row) in enumerate(model_dates.iterrows()):
    dt = row['date']
    if i % 200 == 0:
        print(f'  Processing {dt.date()} ({i}/{len(model_dates)})...')

    # PM prediction
    try:
        pm = pm_predict(dt)
        pm_results.append({
            'date': dt,
            'pm_predicted_shape': pm.get('predicted_shape'),
            'pm_confidence': pm.get('confidence'),
            'pm_shape_probs': pm.get('shape_probs', {}),
        })
    except Exception:
        pm_results.append({'date': dt, 'pm_predicted_shape': None, 'pm_confidence': None, 'pm_shape_probs': {}})

    # TM 1w prediction
    try:
        tm1 = tm_predict(dt, '1w')
        tm_1w_results.append({
            'date': dt,
            'tm_1w_top1_shape': tm1.get('top1_shape'),
            'tm_1w_top1_prob': tm1.get('top1_prob'),
            'tm_1w_top2_shape': tm1.get('top2_shape'),
            'tm_1w_top2_prob': tm1.get('top2_prob'),
            'tm_1w_all_probs': tm1.get('all_probs', {}),
        })
    except Exception:
        tm_1w_results.append({'date': dt, 'tm_1w_top1_shape': None, 'tm_1w_top1_prob': None,
                              'tm_1w_top2_shape': None, 'tm_1w_top2_prob': None, 'tm_1w_all_probs': {}})

    # TM 2w prediction
    try:
        tm2 = tm_predict(dt, '2w')
        tm_2w_results.append({
            'date': dt,
            'tm_2w_top1_shape': tm2.get('top1_shape'),
            'tm_2w_top1_prob': tm2.get('top1_prob'),
        })
    except Exception:
        tm_2w_results.append({'date': dt, 'tm_2w_top1_shape': None, 'tm_2w_top1_prob': None})

pm_df = pd.DataFrame(pm_results)
tm1_df = pd.DataFrame(tm_1w_results)
tm2_df = pd.DataFrame(tm_2w_results)

# Merge model outputs onto main df
df = df.merge(pm_df, on='date', how='left')
df = df.merge(tm1_df, on='date', how='left')
df = df.merge(tm2_df, on='date', how='left')

# Compute PM Level
def compute_pm_level(row):
    if pd.isna(row.get('pm_predicted_shape')) or pd.isna(row.get('pm_confidence')):
        return np.nan
    obs = str(row['shape'])
    pred = str(row['pm_predicted_shape'])
    conf = row['pm_confidence']
    probs = row.get('pm_shape_probs', {})

    if pred == obs and conf >= PM_CONFIDENCE_THRESHOLD:
        return 0  # Level 0: confirmed with high confidence
    # Check if observed is in top-2
    if isinstance(probs, dict) and probs:
        sorted_shapes = sorted(probs.items(), key=lambda x: -x[1])
        top2 = [s for s, _ in sorted_shapes[:2]]
        if obs in top2:
            return 1  # Level 1: observed is top-2
    if pred == obs:  # Correct but low confidence
        return 1
    return 2  # Level 2: observed is top-3 or worse

df['pm_level'] = df.apply(compute_pm_level, axis=1)

# Compute TM 1w cell rating
def get_tm_1w_rating(row):
    cs = str(row['shape'])
    ts = row.get('tm_1w_top1_shape')
    if pd.isna(ts) or ts is None:
        return None
    return TM_1W_RATINGS.get((cs, str(ts)), 'IGNORE')

df['tm_1w_cell_rating'] = df.apply(get_tm_1w_rating, axis=1)

print(f'\nPM Level distribution (2017+):\n{df[df["date"] >= model_start]["pm_level"].value_counts().sort_index()}')
print(f'\nTM 1w cell rating distribution (2017+):\n{df[df["date"] >= model_start]["tm_1w_cell_rating"].value_counts()}')
print('\nModel outputs merged.')

Running PM + TM predictions for 2268 dates (2017+)...
  Processing 2017-01-02 (0/2268)...


  Processing 2018-01-03 (200/2268)...


  Processing 2018-11-07 (400/2268)...


  Processing 2019-09-04 (600/2268)...


  Processing 2020-06-26 (800/2268)...


...

  Processing 2023-10-04 (1600/2268)...


  Processing 2024-07-30 (1800/2268)...


  Processing 2025-05-22 (2000/2268)...


  Processing 2026-03-16 (2200/2268)...



PM Level distribution (2017+):
pm_level
0.0    1958
1.0      41
2.0      17
Name: count, dtype: int64

TM 1w cell rating distribution (2017+):
tm_1w_cell_rating
IGNORE    1580
USE        338
SKIP        98
Name: count, dtype: int64

Model outputs merged.


In [5]:
# ══════════════════════════════════════════════════════════════
# CELL 5 — STRATEGY A: MEAN REVERSION IN SB (0.0) AND C (1)
# ══════════════════════════════════════════════════════════════

def run_strategy_a(df, instrument, z_threshold=2.0, oos_start=OOS_START):
    """
    Run Strategy A signal validity test for one instrument.

    Returns dict with:
      - signal_on/off counts (IS/OOS)
      - reversion rates at 5/10/20 day horizons
      - reversion magnitudes
      - shape survival rates
      - PM level breakdown
      - episode counts
    """
    z_col = f'{instrument}_z'
    work = df.copy()

    # Signal ON (strict): SB or C, ESTABLISHED, |z| > threshold, PM Level 0
    work['sig_a_strict'] = (
        work['shape'].isin(['0.0', '1']) &
        (work['duration_stage'] == 'ESTABLISHED') &
        (work[z_col].abs() > z_threshold) &
        work[z_col].notna() &
        (work['pm_level'] == 0)
    )

    # Signal ON (relaxed): same without PM filter
    work['sig_a_relaxed'] = (
        work['shape'].isin(['0.0', '1']) &
        (work['duration_stage'] == 'ESTABLISHED') &
        (work[z_col].abs() > z_threshold) &
        work[z_col].notna()
    )

    # Forward-looking measurements
    for h in [5, 10, 20]:
        # Future z-score
        work[f'z_t{h}'] = work[z_col].shift(-h)
        # Future shape
        work[f'shape_t{h}'] = work['shape'].shift(-h)

    results = {'instrument': instrument, 'z_threshold': z_threshold}

    for signal_type in ['strict', 'relaxed']:
        sig_col = f'sig_a_{signal_type}'
        for period_name, period_mask in [('IS', work['date'] < oos_start),
                                          ('OOS', work['date'] >= oos_start)]:
            sub = work[period_mask].copy()
            on = sub[sub[sig_col]]
            off = sub[~sub[sig_col]]

            prefix = f'{signal_type}_{period_name}'
            results[f'{prefix}_n_on'] = len(on)
            results[f'{prefix}_n_off'] = len(off)

            # Episode count for Signal ON
            if len(on) > 0:
                ep_changes = (on['episode_id'] != on['episode_id'].shift(1)) | \
                             (~on[sig_col].shift(1, fill_value=False))
                results[f'{prefix}_n_episodes'] = ep_changes.sum()
            else:
                results[f'{prefix}_n_episodes'] = 0

            for h in [5, 10, 20]:
                # Reversion: z-score moved toward zero
                for label, data in [('on', on), ('off', off)]:
                    valid = data[[z_col, f'z_t{h}']].dropna()
                    if len(valid) == 0:
                        results[f'{prefix}_{label}_rev_partial_{h}d'] = np.nan
                        results[f'{prefix}_{label}_rev_full_{h}d'] = np.nan
                        results[f'{prefix}_{label}_z_chg_{h}d'] = np.nan
                        continue

                    z_entry = valid[z_col].abs()
                    z_future = valid[f'z_t{h}'].abs()

                    # Partial reversion: |z| decreased
                    partial = (z_future < z_entry).mean() * 100
                    results[f'{prefix}_{label}_rev_partial_{h}d'] = round(partial, 1)

                    # Full reversion: |z| at t+h < 0.5
                    full = (z_future < 0.5).mean() * 100
                    results[f'{prefix}_{label}_rev_full_{h}d'] = round(full, 1)

                    # Magnitude: average change in |z| (negative = reversion)
                    z_chg = (z_future - z_entry).mean()
                    results[f'{prefix}_{label}_z_chg_{h}d'] = round(z_chg, 3)

                # Shape survival (same for all instruments — compute anyway)
                valid_shape = on[['shape', f'shape_t{h}']].dropna()
                if len(valid_shape) > 0:
                    surv = (valid_shape['shape'] == valid_shape[f'shape_t{h}']).mean() * 100
                    results[f'{prefix}_shape_survival_{h}d'] = round(surv, 1)
                else:
                    results[f'{prefix}_shape_survival_{h}d'] = np.nan

    # PM Level breakdown for strict OOS Signal ON days
    oos_strict = work[(work['date'] >= oos_start) & work['sig_a_relaxed']]
    for lvl in [0, 1, 2]:
        lvl_data = oos_strict[oos_strict['pm_level'] == lvl]
        results[f'oos_pm_lvl{lvl}_n'] = len(lvl_data)
        valid = lvl_data[[z_col, 'z_t10']].dropna()
        if len(valid) > 0:
            rev = (valid['z_t10'].abs() < valid[z_col].abs()).mean() * 100
            results[f'oos_pm_lvl{lvl}_rev_10d'] = round(rev, 1)
        else:
            results[f'oos_pm_lvl{lvl}_rev_10d'] = np.nan

    return results


# Run Strategy A across all 9 instruments at multiple z-thresholds
print('Running Strategy A...')
strat_a_results = []
for inst in STRAT_A_INSTRUMENTS:
    for zt in [1.5, 2.0, 2.5, 3.0]:
        r = run_strategy_a(df, inst, z_threshold=zt)
        strat_a_results.append(r)
        if zt == 2.0:
            n_on = r['strict_OOS_n_on']
            n_ep = r['strict_OOS_n_episodes']
            rev_on = r.get('strict_OOS_on_rev_partial_10d', '-')
            rev_off = r.get('strict_OOS_off_rev_partial_10d', '-')
            print(f'  {inst} (z=2.0): OOS Signal ON={n_on} days ({n_ep} episodes), '
                  f'rev@10d ON={rev_on}% OFF={rev_off}%')

strat_a_df = pd.DataFrame(strat_a_results)
print(f'\nStrategy A complete: {len(strat_a_df)} rows ({len(STRAT_A_INSTRUMENTS)} instruments x 4 z-thresholds)')

Running Strategy A...
  M1-M2 (z=2.0): OOS Signal ON=4 days (2 episodes), rev@10d ON=100.0% OFF=51.5%


  M2-M3 (z=2.0): OOS Signal ON=10 days (3 episodes), rev@10d ON=100.0% OFF=51.9%
  M3-M4 (z=2.0): OOS Signal ON=10 days (5 episodes), rev@10d ON=100.0% OFF=52.8%


  M4-M5 (z=2.0): OOS Signal ON=14 days (3 episodes), rev@10d ON=85.7% OFF=48.8%
  M5-M6 (z=2.0): OOS Signal ON=19 days (3 episodes), rev@10d ON=100.0% OFF=47.9%


  BF_M1M2M3 (z=2.0): OOS Signal ON=8 days (3 episodes), rev@10d ON=85.7% OFF=54.7%
  BF_M2M3M4 (z=2.0): OOS Signal ON=6 days (3 episodes), rev@10d ON=100.0% OFF=51.9%


  BF_M3M4M5 (z=2.0): OOS Signal ON=13 days (4 episodes), rev@10d ON=50.0% OFF=49.5%
  BF_M4M5M6 (z=2.0): OOS Signal ON=6 days (4 episodes), rev@10d ON=100.0% OFF=45.7%



Strategy A complete: 36 rows (9 instruments x 4 z-thresholds)


In [6]:
# ══════════════════════════════════════════════════════════════
# CELL 6 — STRATEGY B: TREND FOLLOWING IN MB, TB, F
# ══════════════════════════════════════════════════════════════

# Direction mapping: (current_shape, target_shape) -> expected direction
# 'WIDEN' = near - far increases, 'NARROW' = near - far decreases
DIRECTION_MAP = {
    ('0.1', '0.0'): 'WIDEN',    # MB -> SB: more backwardated
    ('0.1', '2'):   'NARROW',   # MB -> F: flatten
    ('0.1', '1'):   'NARROW',   # MB -> C: toward contango
    ('0.2', '0.0'): 'WIDEN',    # TB -> SB: more backwardated
    ('0.2', '1'):   'NARROW',   # TB -> C: toward contango
    ('2', '0.1'):   'WIDEN',    # F -> MB: widen from flat
    ('2', '1'):     'NARROW',   # F -> C: toward contango
}


def run_strategy_b(df, pair_name, near_col, far_col, oos_start=OOS_START):
    """
    Run Strategy B signal validity test for one calendar pair.

    Returns dict with directional hit rates, TM agreement analysis, etc.
    """
    spread_col = pair_name
    work = df.copy()

    # Signal ON: transitional shape (MB, TB, F), EARLY or SETTLING,
    # TM 1w predicts a specific target, cell rated USE
    work['is_transitional'] = work['shape'].isin(['0.1', '0.2', '2'])
    work['is_early_settling'] = work['duration_stage'].isin(['EARLY', 'SETTLING'])
    work['has_tm_prediction'] = work['tm_1w_top1_shape'].notna()
    work['cell_is_use'] = work['tm_1w_cell_rating'] == 'USE'

    # Determine predicted direction for each day
    def get_predicted_direction(row):
        cs = str(row['shape'])
        ts = row.get('tm_1w_top1_shape')
        if pd.isna(ts) or ts is None:
            return None
        return DIRECTION_MAP.get((cs, str(ts)))

    work['predicted_direction'] = work.apply(get_predicted_direction, axis=1)

    work['sig_b_on'] = (
        work['is_transitional'] &
        work['is_early_settling'] &
        work['has_tm_prediction'] &
        work['cell_is_use'] &
        work['predicted_direction'].notna()
    )

    # Forward spread changes
    for h in [5, 10]:
        work[f'spread_t{h}'] = work[spread_col].shift(-h)
        work[f'spread_chg_{h}d'] = work[f'spread_t{h}'] - work[spread_col]
        # Future shape (curve-wide)
        work[f'shape_t{h}'] = work['shape'].shift(-h)

    # Directional hit: did spread move in predicted direction?
    MIN_TICK = 1.0  # minimum move to count as directional
    for h in [5, 10]:
        def dir_hit(row, horizon=h):
            d = row.get('predicted_direction')
            chg = row.get(f'spread_chg_{horizon}d')
            if d is None or pd.isna(chg):
                return np.nan
            if d == 'WIDEN':
                return 1 if chg >= MIN_TICK else 0
            elif d == 'NARROW':
                return 1 if chg <= -MIN_TICK else 0
            return np.nan
        work[f'dir_hit_{h}d'] = work.apply(dir_hit, axis=1)

    results = {'pair': pair_name}

    for period_name, period_mask in [('IS', work['date'] < oos_start),
                                      ('OOS', work['date'] >= oos_start)]:
        sub = work[period_mask]
        on = sub[sub['sig_b_on']]
        # Signal OFF: transitional but not Signal ON
        off = sub[sub['is_transitional'] & ~sub['sig_b_on']]

        results[f'{period_name}_n_on'] = len(on)
        results[f'{period_name}_n_off'] = len(off)

        # Episode count
        if len(on) > 0:
            ep_changes = (on['episode_id'] != on['episode_id'].shift(1)) | \
                         (~on['sig_b_on'].shift(1, fill_value=False))
            results[f'{period_name}_n_episodes'] = ep_changes.sum()
        else:
            results[f'{period_name}_n_episodes'] = 0

        for h in [5, 10]:
            for label, data in [('on', on), ('off', off)]:
                hits = data[f'dir_hit_{h}d'].dropna()
                if len(hits) > 0:
                    results[f'{period_name}_{label}_hit_{h}d'] = round(hits.mean() * 100, 1)
                    results[f'{period_name}_{label}_hit_{h}d_n'] = len(hits)
                else:
                    results[f'{period_name}_{label}_hit_{h}d'] = np.nan
                    results[f'{period_name}_{label}_hit_{h}d_n'] = 0

                # Average spread move magnitude
                for dirctn in ['WIDEN', 'NARROW']:
                    dir_data = data[data['predicted_direction'] == dirctn][f'spread_chg_{h}d'].dropna()
                    if len(dir_data) > 0:
                        results[f'{period_name}_{label}_{dirctn}_avg_chg_{h}d'] = round(dir_data.mean(), 1)

            # Target shape confirmation (curve-wide, reuse across pairs)
            on_valid = on[['tm_1w_top1_shape', f'shape_t{h}']].dropna()
            if len(on_valid) > 0:
                confirmed = (on_valid['tm_1w_top1_shape'] == on_valid[f'shape_t{h}']).mean() * 100
                results[f'{period_name}_target_confirm_{h}d'] = round(confirmed, 1)
            else:
                results[f'{period_name}_target_confirm_{h}d'] = np.nan

    # TM 1w/2w agreement vs disagreement (OOS only)
    oos_on = work[(work['date'] >= oos_start) & work['sig_b_on']].copy()
    if len(oos_on) > 0:
        oos_on['tm_agree'] = oos_on['tm_1w_top1_shape'] == oos_on['tm_2w_top1_shape']
        for agree_label, agree_val in [('agree', True), ('disagree', False)]:
            ag_data = oos_on[oos_on['tm_agree'] == agree_val]
            results[f'OOS_{agree_label}_n'] = len(ag_data)
            hits_5 = ag_data['dir_hit_5d'].dropna()
            results[f'OOS_{agree_label}_hit_5d'] = round(hits_5.mean() * 100, 1) if len(hits_5) > 0 else np.nan

    return results


# Run Strategy B across all 5 calendar pairs
print('Running Strategy B...')
strat_b_results = []
for pair_name, near, far in CALENDAR_PAIRS:
    r = run_strategy_b(df, pair_name, near, far)
    strat_b_results.append(r)
    n_on = r['OOS_n_on']
    n_ep = r['OOS_n_episodes']
    hit_on = r.get('OOS_on_hit_5d', '-')
    hit_off = r.get('OOS_off_hit_5d', '-')
    print(f'  {pair_name}: OOS Signal ON={n_on} days ({n_ep} episodes), '
          f'hit@5d ON={hit_on}% OFF={hit_off}%')

strat_b_df = pd.DataFrame(strat_b_results)
print(f'\nStrategy B complete: {len(strat_b_df)} rows')

Running Strategy B...
  M1-M2: OOS Signal ON=50 days (19 episodes), hit@5d ON=44.0% OFF=100.0%


  M2-M3: OOS Signal ON=50 days (19 episodes), hit@5d ON=48.0% OFF=100.0%
  M3-M4: OOS Signal ON=50 days (19 episodes), hit@5d ON=44.0% OFF=0.0%
  M4-M5: OOS Signal ON=50 days (19 episodes), hit@5d ON=40.0% OFF=0.0%


  M5-M6: OOS Signal ON=50 days (19 episodes), hit@5d ON=46.0% OFF=0.0%

Strategy B complete: 5 rows


In [7]:
# ══════════════════════════════════════════════════════════════
# CELL 7 — SIGNAL CORRELATION + SAMPLE SIZE CHECKS
# ══════════════════════════════════════════════════════════════

# ── Signal correlation across adjacent pairs (Strategy A, z=2.0) ──
print('Signal ON correlation across calendar pairs (Strategy A, z=2.0, OOS):')
oos = df[df['date'] >= OOS_START].copy()

# Build signal ON matrix for each pair
sig_on_cols = {}
for inst in [p[0] for p in CALENDAR_PAIRS]:
    z_col = f'{inst}_z'
    sig_on_cols[inst] = (
        oos['shape'].isin(['0.0', '1']) &
        (oos['duration_stage'] == 'ESTABLISHED') &
        (oos[z_col].abs() > 2.0) &
        oos[z_col].notna() &
        (oos['pm_level'] == 0)
    ).astype(int)

sig_matrix = pd.DataFrame(sig_on_cols)
print(sig_matrix.corr().round(2).to_string())

# Count overlap: how many days are Signal ON for multiple pairs simultaneously
sig_sum = sig_matrix.sum(axis=1)
print(f'\nSignal ON day overlap (how many pairs fire simultaneously):')
for n_pairs in range(1, 6):
    count = (sig_sum == n_pairs).sum()
    if count > 0:
        print(f'  {n_pairs} pair(s) ON simultaneously: {count} days')
print(f'  Zero pairs ON: {(sig_sum == 0).sum()} days')

# ── Sample size checks ──────────────────────────────────────
print('\n' + '='*60)
print('SAMPLE SIZE CHECKS')
print('='*60)

print('\nStrategy A — Signal ON counts by shape (z=2.0, strict, OOS):')
for inst in STRAT_A_INSTRUMENTS:
    row = strat_a_df[(strat_a_df['instrument'] == inst) & (strat_a_df['z_threshold'] == 2.0)].iloc[0]
    n = row['strict_OOS_n_on']
    ep = row['strict_OOS_n_episodes']
    flag = '  ** INSUFFICIENT' if n < 10 else ('  * directional only' if n < 30 else '')
    print(f'  {inst:12s}: n={n:4d} ({ep} episodes){flag}')

print('\nStrategy B — Signal ON counts by pair (OOS):')
for _, row in strat_b_df.iterrows():
    n = row['OOS_n_on']
    ep = row['OOS_n_episodes']
    flag = '  ** INSUFFICIENT' if n < 10 else ('  * directional only' if n < 30 else '')
    print(f'  {row["pair"]:12s}: n={n:4d} ({ep} episodes){flag}')

Signal ON correlation across calendar pairs (Strategy A, z=2.0, OOS):
       M1-M2  M2-M3  M3-M4  M4-M5  M5-M6
M1-M2   1.00  -0.01  -0.01  -0.01   0.11
M2-M3  -0.01   1.00   0.50   0.25  -0.01
M3-M4  -0.01   0.50   1.00   0.42   0.13
M4-M5  -0.01   0.25   0.42   1.00   0.48
M5-M6   0.11  -0.01   0.13   0.48   1.00

Signal ON day overlap (how many pairs fire simultaneously):
  1 pair(s) ON simultaneously: 24 days
  2 pair(s) ON simultaneously: 9 days
  3 pair(s) ON simultaneously: 5 days
  Zero pairs ON: 1056 days

SAMPLE SIZE CHECKS

Strategy A — Signal ON counts by shape (z=2.0, strict, OOS):
  M1-M2       : n=   4 (2 episodes)  ** INSUFFICIENT
  M2-M3       : n=  10 (3 episodes)  * directional only
  M3-M4       : n=  10 (5 episodes)  * directional only
  M4-M5       : n=  14 (3 episodes)  * directional only
  M5-M6       : n=  19 (3 episodes)  * directional only
  BF_M1M2M3   : n=   8 (3 episodes)  ** INSUFFICIENT
  BF_M2M3M4   : n=   6 (3 episodes)  ** INSUFFICIENT
  BF_M3M4M5   : 

In [8]:
# ══════════════════════════════════════════════════════════════
# CELL 8 — GATE CHECK + LOGGING
# ══════════════════════════════════════════════════════════════

output_lines = [
    '',
    '=' * 70,
    f'STAGE 0 — SIGNAL VALIDITY TEST — {datetime.now().strftime("%Y-%m-%d %H:%M")}',
    '=' * 70,
    '',
]

# ── Data summary ──────────────────────────────────────────────
output_lines.append('--- DATA SUMMARY ---')
output_lines.append(f'Date range: {df["date"].min().date()} to {df["date"].max().date()}')
output_lines.append(f'Total trading days: {len(df)}')
output_lines.append(f'Model outputs available: 2017-01-02 to {df[df["pm_predicted_shape"].notna()]["date"].max().date()}')
output_lines.append(f'In-sample: < {OOS_START} | Out-of-sample: >= {OOS_START}')
output_lines.append(f'Shape distribution (full panel):')
for s in sorted(df['shape'].unique()):
    n = (df['shape'] == s).sum()
    output_lines.append(f'  Shape {s}: {n} days ({n/len(df)*100:.1f}%)')
output_lines.append(f'Instruments tested:')
output_lines.append(f'  Calendar pairs: {", ".join(STRAT_B_INSTRUMENTS)}')
output_lines.append(f'  Butterflies: {", ".join([b[0] for b in BUTTERFLIES])}')

# ── Strategy A results ────────────────────────────────────────
output_lines.append('')
output_lines.append('--- STRATEGY A: MEAN REVERSION (SB, C) ---')
output_lines.append('Signal ON = ESTABLISHED (20+ days) + |z| > threshold + PM Level 0')
output_lines.append('Reversion = |z| at t+N < |z| at entry (partial) or |z| < 0.5 (full)')
output_lines.append('')

for inst in STRAT_A_INSTRUMENTS:
    output_lines.append(f'  [{inst}]')
    for zt in [1.5, 2.0, 2.5, 3.0]:
        row = strat_a_df[(strat_a_df['instrument'] == inst) & (strat_a_df['z_threshold'] == zt)].iloc[0]

        # OOS strict results
        n_on = row['strict_OOS_n_on']
        n_off = row['strict_OOS_n_off']
        n_ep = row['strict_OOS_n_episodes']
        rev_on_10 = row.get('strict_OOS_on_rev_partial_10d', '-')
        rev_off_10 = row.get('strict_OOS_off_rev_partial_10d', '-')
        z_chg_on_10 = row.get('strict_OOS_on_z_chg_10d', '-')

        # OOS relaxed results
        n_on_r = row['relaxed_OOS_n_on']
        rev_on_r_10 = row.get('relaxed_OOS_on_rev_partial_10d', '-')

        output_lines.append(f'    z={zt}: OOS strict n={n_on} ({n_ep} ep) '
                          f'rev@10d ON={rev_on_10}% OFF={rev_off_10}% '
                          f'z_chg={z_chg_on_10} | '
                          f'relaxed n={n_on_r} rev@10d={rev_on_r_10}%')

    # Detail at z=2.0 for primary horizon
    row20 = strat_a_df[(strat_a_df['instrument'] == inst) & (strat_a_df['z_threshold'] == 2.0)].iloc[0]
    for h in [5, 10, 20]:
        on_p = row20.get(f'strict_OOS_on_rev_partial_{h}d', '-')
        off_p = row20.get(f'strict_OOS_off_rev_partial_{h}d', '-')
        on_f = row20.get(f'strict_OOS_on_rev_full_{h}d', '-')
        off_f = row20.get(f'strict_OOS_off_rev_full_{h}d', '-')
        surv = row20.get(f'strict_OOS_shape_survival_{h}d', '-')
        output_lines.append(f'    z=2.0 @{h}d: partial ON={on_p}% OFF={off_p}%  '
                          f'full ON={on_f}% OFF={off_f}%  shape_surv={surv}%')

    # PM level breakdown
    for lvl in [0, 1, 2]:
        n_lvl = row20.get(f'oos_pm_lvl{lvl}_n', 0)
        rev_lvl = row20.get(f'oos_pm_lvl{lvl}_rev_10d', '-')
        output_lines.append(f'    PM Level {lvl}: n={n_lvl} rev@10d={rev_lvl}%')

    output_lines.append('')

# Strategy A cross-instrument summary (z=2.0, OOS, 10d)
output_lines.append('  STRATEGY A — CROSS-INSTRUMENT SUMMARY (z=2.0, OOS, 10d horizon):')
output_lines.append(f'  {"Instrument":<14} {"n ON":>6} {"n ep":>5} {"rev ON":>8} {"rev OFF":>8} {"delta":>7} {"Gate":>12}')
output_lines.append(f'  {"-"*64}')

strat_a_gates = []
for inst in STRAT_A_INSTRUMENTS:
    row = strat_a_df[(strat_a_df['instrument'] == inst) & (strat_a_df['z_threshold'] == 2.0)].iloc[0]
    n_on = row['strict_OOS_n_on']
    n_ep = row['strict_OOS_n_episodes']
    rev_on = row.get('strict_OOS_on_rev_partial_10d', np.nan)
    rev_off = row.get('strict_OOS_off_rev_partial_10d', np.nan)

    if n_on < 10:
        gate = 'INSUFFICIENT'
    elif n_on < 30:
        if not np.isnan(rev_on) and not np.isnan(rev_off) and (rev_on - rev_off) > 5:
            gate = 'PASS*'  # Directional only
        else:
            gate = 'INSUFFICIENT'
    else:
        if not np.isnan(rev_on) and not np.isnan(rev_off) and (rev_on - rev_off) > 5:
            gate = 'PASS'
        else:
            gate = 'FAIL'

    delta = round(rev_on - rev_off, 1) if not np.isnan(rev_on) and not np.isnan(rev_off) else '-'
    rev_on_s = f'{rev_on:.1f}%' if not np.isnan(rev_on) else '-'
    rev_off_s = f'{rev_off:.1f}%' if not np.isnan(rev_off) else '-'

    output_lines.append(f'  {inst:<14} {n_on:>6} {n_ep:>5} {rev_on_s:>8} {rev_off_s:>8} {str(delta)+"pp":>7} {gate:>12}')
    strat_a_gates.append({'instrument': inst, 'gate': gate, 'n_on': n_on, 'delta': delta})

# ── Strategy B results ────────────────────────────────────────
output_lines.append('')
output_lines.append('--- STRATEGY B: TREND FOLLOWING (MB, TB, F) ---')
output_lines.append('Signal ON = transitional shape + EARLY/SETTLING + TM 1w cell rated USE')
output_lines.append('Direction per DIRECTION_MAP; hit = spread moved >= 1 tick in predicted direction')
output_lines.append('')

for _, row in strat_b_df.iterrows():
    pair = row['pair']
    output_lines.append(f'  [{pair}]')

    for period in ['IS', 'OOS']:
        n_on = row[f'{period}_n_on']
        n_off = row[f'{period}_n_off']
        n_ep = row[f'{period}_n_episodes']
        output_lines.append(f'    {period}: Signal ON={n_on} ({n_ep} ep), OFF={n_off}')
        for h in [5, 10]:
            hit_on = row.get(f'{period}_on_hit_{h}d', '-')
            hit_off = row.get(f'{period}_off_hit_{h}d', '-')
            hit_on_n = row.get(f'{period}_on_hit_{h}d_n', 0)
            hit_off_n = row.get(f'{period}_off_hit_{h}d_n', 0)
            confirm = row.get(f'{period}_target_confirm_{h}d', '-')
            output_lines.append(f'      @{h}d: ON hit={hit_on}% (n={hit_on_n}) '
                              f'OFF hit={hit_off}% (n={hit_off_n}) '
                              f'target_confirm={confirm}%')

    # TM agreement
    for ag in ['agree', 'disagree']:
        n_ag = row.get(f'OOS_{ag}_n', 0)
        hit_ag = row.get(f'OOS_{ag}_hit_5d', '-')
        output_lines.append(f'    TM 1w/2w {ag}: n={n_ag} hit@5d={hit_ag}%')
    output_lines.append('')

# Strategy B cross-pair summary
output_lines.append('  STRATEGY B — CROSS-PAIR SUMMARY (OOS, 5d horizon):')
output_lines.append(f'  {"Pair":<14} {"n ON":>6} {"n ep":>5} {"hit ON":>8} {"hit OFF":>8} {"delta":>7} {"Gate":>12}')
output_lines.append(f'  {"-"*64}')

strat_b_gates = []
for _, row in strat_b_df.iterrows():
    pair = row['pair']
    n_on = row['OOS_n_on']
    n_ep = row['OOS_n_episodes']
    hit_on = row.get('OOS_on_hit_5d', np.nan)
    hit_off = row.get('OOS_off_hit_5d', np.nan)

    if n_on < 10:
        gate = 'INSUFFICIENT'
    elif n_on < 30:
        if not np.isnan(hit_on) and not np.isnan(hit_off) and (hit_on - hit_off) > 5:
            gate = 'PASS*'
        else:
            gate = 'INSUFFICIENT'
    else:
        if not np.isnan(hit_on) and not np.isnan(hit_off) and (hit_on - hit_off) > 5:
            gate = 'PASS'
        else:
            gate = 'FAIL'

    delta = round(hit_on - hit_off, 1) if not np.isnan(hit_on) and not np.isnan(hit_off) else '-'
    hit_on_s = f'{hit_on:.1f}%' if not np.isnan(hit_on) else '-'
    hit_off_s = f'{hit_off:.1f}%' if not np.isnan(hit_off) else '-'

    output_lines.append(f'  {pair:<14} {n_on:>6} {n_ep:>5} {hit_on_s:>8} {hit_off_s:>8} {str(delta)+"pp":>7} {gate:>12}')
    strat_b_gates.append({'pair': pair, 'gate': gate, 'n_on': n_on, 'delta': delta})

# ── Signal correlation note ───────────────────────────────────
output_lines.append('')
output_lines.append('--- SIGNAL CORRELATION (Strategy A, z=2.0, calendar pairs, OOS) ---')
output_lines.append(sig_matrix.corr().round(2).to_string())
output_lines.append(f'Day overlap: {(sig_sum >= 2).sum()} days with 2+ pairs ON simultaneously '
                   f'out of {(sig_sum >= 1).sum()} total Signal ON days')

# ── Overall gate verdict ──────────────────────────────────────
output_lines.append('')
output_lines.append('--- GATE CHECK VERDICT ---')

a_passes = [g for g in strat_a_gates if g['gate'] in ('PASS', 'PASS*')]
a_fails = [g for g in strat_a_gates if g['gate'] == 'FAIL']
a_insuf = [g for g in strat_a_gates if g['gate'] == 'INSUFFICIENT']

output_lines.append(f'Strategy A: {len(a_passes)} PASS, {len(a_fails)} FAIL, {len(a_insuf)} INSUFFICIENT')
if a_passes:
    output_lines.append(f'  Passing instruments: {", ".join(g["instrument"] for g in a_passes)}')
    output_lines.append(f'  -> PROCEED to Stage 1 for these instruments')
else:
    output_lines.append(f'  -> NO passing instruments. Investigate before proceeding.')

b_passes = [g for g in strat_b_gates if g['gate'] in ('PASS', 'PASS*')]
b_fails = [g for g in strat_b_gates if g['gate'] == 'FAIL']
b_insuf = [g for g in strat_b_gates if g['gate'] == 'INSUFFICIENT']

output_lines.append(f'Strategy B: {len(b_passes)} PASS, {len(b_fails)} FAIL, {len(b_insuf)} INSUFFICIENT')
if b_passes:
    output_lines.append(f'  Passing pairs: {", ".join(g["pair"] for g in b_passes)}')
    if a_passes:
        output_lines.append(f'  -> PROCEED to Stage 1 for these pairs (Strategy A has >=1 pass)')
    else:
        output_lines.append(f'  -> HOLD: Strategy A must pass first per sequencing rule')
else:
    output_lines.append(f'  -> NO passing pairs.')

# Write to log
content = '\n'.join(output_lines)
print(content)

with open(LOG_FILE, 'a', encoding='utf-8') as f:
    f.write('\n' + content + '\n')
print(f'\n[Appended to {LOG_FILE}]')


STAGE 0 — SIGNAL VALIDITY TEST — 2026-07-10 13:38

--- DATA SUMMARY ---
Date range: 2008-01-01 to 2026-06-26
Total trading days: 4070
Model outputs available: 2017-01-02 to 2026-06-26
In-sample: < 2022-01-01 | Out-of-sample: >= 2022-01-01
Shape distribution (full panel):
  Shape 0.0: 1330 days (32.7%)
  Shape 0.1: 588 days (14.4%)
  Shape 0.2: 372 days (9.1%)
  Shape 1: 1219 days (30.0%)
  Shape 2: 561 days (13.8%)
Instruments tested:
  Calendar pairs: M1-M2, M2-M3, M3-M4, M4-M5, M5-M6
  Butterflies: BF_M1M2M3, BF_M2M3M4, BF_M3M4M5, BF_M4M5M6

--- STRATEGY A: MEAN REVERSION (SB, C) ---
Signal ON = ESTABLISHED (20+ days) + |z| > threshold + PM Level 0
Reversion = |z| at t+N < |z| at entry (partial) or |z| < 0.5 (full)

  [M1-M2]
    z=1.5: OOS strict n=22 (7 ep) rev@10d ON=93.8% OFF=50.0% z_chg=-0.973 | relaxed n=22 rev@10d=93.8%
    z=2.0: OOS strict n=4 (2 ep) rev@10d ON=100.0% OFF=51.5% z_chg=-1.629 | relaxed n=4 rev@10d=100.0%
    z=2.5: OOS strict n=2 (2 ep) rev@10d ON=100.0% OFF=

In [9]:
# ══════════════════════════════════════════════════════════════
# CELL 9 — TF SIGNAL OFF FIX & RERUN
# ══════════════════════════════════════════════════════════════
# DIAGNOSIS: Signal OFF days lack predicted_direction because:
#   - OFF days are transitional but failed condition (d) cell=USE
#     or (e) no direction mapping (e.g. F->F is USE but has no direction)
#   - Without predicted_direction, dir_hit returns NaN -> n=0 effective
#
# ROOT CAUSE BREAKDOWN (OOS transitional days = 456):
#   - ESTABLISHED: 31 days (no TM needed for these)
#   - EARLY/SETTLING, no TM pred available: ~0 (TM predicts for all)
#   - EARLY/SETTLING, has pred, cell not USE: 193 days
#   - EARLY/SETTLING, has pred, cell USE but no direction map: 182 days
#     (mostly F->F: USE-rated but F staying F has no directional implication)
#   - Signal ON (all conditions met): 50 days
#
# CALIBRATION RATING DISTRIBUTION (1w, transitional shapes):
#   MB (0.1): 0.0=USE, 0.1=IGNORE, 0.2=IGNORE, 1=USE, 2=IGNORE -> 2 USE, 3 non-USE
#   TB (0.2): ALL SKIP (5 cells)
#   F  (2):   0.0=IGNORE, 0.1=IGNORE, 0.2=IGNORE, 1=USE, 2=USE -> 2 USE, 3 non-USE
#   Total: 4 USE, 11 non-USE out of 15 transitional cells (but 5 are SKIP)
#
# FIX CHOSEN: Option 3 (unconditional baseline)
# Signal OFF = ALL transitional days (regardless of duration/rating) where
# TM has a prediction AND the direction can be derived from TM top-1.
# This removes conditions (b), (d) from OFF, keeping only (a) + (c) + (e).
# Signal ON still requires all 5 conditions. The gate tests whether the
# USE + EARLY/SETTLING filter improves hit rate over unfiltered TM predictions
# on transitional days.
#
# Additionally, for OFF days where TM predicts a transition that IS in
# DIRECTION_MAP but the cell is IGNORE/SOFT (not USE), we can still compute
# directional hit — the question becomes "does USE filtering help?"

def run_tf_fixed(df, pair_name, near_col, far_col, oos_start=OOS_START):
    """
    Run TF (Strategy B) with corrected Signal OFF definition.

    Signal ON: unchanged (transitional + EARLY/SETTLING + USE + has direction)
    Signal OFF: ALL transitional days with TM prediction and direction mapping,
                excluding Signal ON days. Removes duration and USE filters from OFF.
    """
    spread_col = pair_name
    work = df.copy()

    work['is_transitional'] = work['shape'].isin(['0.1', '0.2', '2'])
    work['is_early_settling'] = work['duration_stage'].isin(['EARLY', 'SETTLING'])
    work['has_tm_prediction'] = work['tm_1w_top1_shape'].notna()
    work['cell_is_use'] = work['tm_1w_cell_rating'] == 'USE'

    def get_predicted_direction(row):
        cs = str(row['shape'])
        ts = row.get('tm_1w_top1_shape')
        if pd.isna(ts) or ts is None:
            return None
        return DIRECTION_MAP.get((cs, str(ts)))

    work['predicted_direction'] = work.apply(get_predicted_direction, axis=1)

    # Signal ON: strict (unchanged)
    work['sig_tf_on'] = (
        work['is_transitional'] &
        work['is_early_settling'] &
        work['has_tm_prediction'] &
        work['cell_is_use'] &
        work['predicted_direction'].notna()
    )

    # Signal OFF (FIXED): transitional + has TM prediction + has direction mapping
    # but NOT Signal ON (i.e. fails duration or USE filter)
    work['has_direction'] = work['predicted_direction'].notna()
    work['sig_tf_off'] = (
        work['is_transitional'] &
        work['has_tm_prediction'] &
        work['has_direction'] &
        ~work['sig_tf_on']
    )

    # Forward spread changes
    MIN_TICK = 1.0
    for h in [5, 10]:
        work[f'spread_t{h}'] = work[spread_col].shift(-h)
        work[f'spread_chg_{h}d'] = work[f'spread_t{h}'] - work[spread_col]
        work[f'shape_t{h}'] = work['shape'].shift(-h)

    for h in [5, 10]:
        def dir_hit(row, horizon=h):
            d = row.get('predicted_direction')
            chg = row.get(f'spread_chg_{horizon}d')
            if d is None or pd.isna(chg):
                return np.nan
            if d == 'WIDEN':
                return 1 if chg >= MIN_TICK else 0
            elif d == 'NARROW':
                return 1 if chg <= -MIN_TICK else 0
            return np.nan
        work[f'dir_hit_{h}d'] = work.apply(dir_hit, axis=1)

    results = {'pair': pair_name}

    for period_name, period_mask in [('IS', work['date'] < oos_start),
                                      ('OOS', work['date'] >= oos_start)]:
        sub = work[period_mask]
        on = sub[sub['sig_tf_on']]
        off = sub[sub['sig_tf_off']]

        results[f'{period_name}_n_on'] = len(on)
        results[f'{period_name}_n_off'] = len(off)

        # Episode count
        if len(on) > 0:
            ep_changes = (on['episode_id'] != on['episode_id'].shift(1)) | \
                         (~on['sig_tf_on'].shift(1, fill_value=False))
            results[f'{period_name}_n_episodes_on'] = ep_changes.sum()
        else:
            results[f'{period_name}_n_episodes_on'] = 0

        if len(off) > 0:
            ep_changes_off = (off['episode_id'] != off['episode_id'].shift(1)) | \
                             (~off['sig_tf_off'].shift(1, fill_value=False))
            results[f'{period_name}_n_episodes_off'] = ep_changes_off.sum()
        else:
            results[f'{period_name}_n_episodes_off'] = 0

        for h in [5, 10]:
            for label, data in [('on', on), ('off', off)]:
                hits = data[f'dir_hit_{h}d'].dropna()
                if len(hits) > 0:
                    results[f'{period_name}_{label}_hit_{h}d'] = round(hits.mean() * 100, 1)
                    results[f'{period_name}_{label}_hit_{h}d_n'] = len(hits)
                else:
                    results[f'{period_name}_{label}_hit_{h}d'] = np.nan
                    results[f'{period_name}_{label}_hit_{h}d_n'] = 0

            # Target shape confirmation
            on_valid = on[['tm_1w_top1_shape', f'shape_t{h}']].dropna()
            if len(on_valid) > 0:
                confirmed = (on_valid['tm_1w_top1_shape'] == on_valid[f'shape_t{h}']).mean() * 100
                results[f'{period_name}_target_confirm_{h}d'] = round(confirmed, 1)
            else:
                results[f'{period_name}_target_confirm_{h}d'] = np.nan

    # TM 1w/2w agreement (OOS only)
    oos_on = work[(work['date'] >= oos_start) & work['sig_tf_on']].copy()
    if len(oos_on) > 0:
        oos_on['tm_agree'] = oos_on['tm_1w_top1_shape'] == oos_on['tm_2w_top1_shape']
        for agree_label, agree_val in [('agree', True), ('disagree', False)]:
            ag_data = oos_on[oos_on['tm_agree'] == agree_val]
            results[f'OOS_{agree_label}_n'] = len(ag_data)
            hits_5 = ag_data['dir_hit_5d'].dropna()
            results[f'OOS_{agree_label}_hit_5d'] = round(hits_5.mean() * 100, 1) if len(hits_5) > 0 else np.nan
    else:
        for ag in ['agree', 'disagree']:
            results[f'OOS_{ag}_n'] = 0
            results[f'OOS_{ag}_hit_5d'] = np.nan

    # OFF breakdown by exclusion reason
    oos_off = work[(work['date'] >= oos_start) & work['sig_tf_off']]
    results['OOS_off_established'] = (oos_off['duration_stage'] == 'ESTABLISHED').sum()
    results['OOS_off_not_use'] = ((oos_off['cell_is_use'] == False) & oos_off['is_early_settling']).sum()
    results['OOS_off_early_settle_use_but_off'] = len(oos_off) - results['OOS_off_established'] - results['OOS_off_not_use']

    return results


# Run TF fixed across all 5 calendar pairs
print('Running TF (fixed Signal OFF)...')
tf_fixed_results = []
for pair_name, near, far in CALENDAR_PAIRS:
    r = run_tf_fixed(df, pair_name, near, far)
    tf_fixed_results.append(r)
    n_on = r['OOS_n_on']
    n_off = r['OOS_n_off']
    n_ep_on = r['OOS_n_episodes_on']
    n_ep_off = r['OOS_n_episodes_off']
    hit_on = r.get('OOS_on_hit_5d', '-')
    hit_off = r.get('OOS_off_hit_5d', '-')
    hit_on_n = r.get('OOS_on_hit_5d_n', 0)
    hit_off_n = r.get('OOS_off_hit_5d_n', 0)
    print(f'  {pair_name}: ON={n_on} ({n_ep_on} ep), OFF={n_off} ({n_ep_off} ep), '
          f'hit@5d ON={hit_on}% (n={hit_on_n}) OFF={hit_off}% (n={hit_off_n})')

tf_fixed_df = pd.DataFrame(tf_fixed_results)

# ── Build comparison table and log ────────────────────────────
log_lines = [
    '',
    '=' * 70,
    f'STAGE 0 — TF SIGNAL OFF FIX & RERUN — {datetime.now().strftime("%Y-%m-%d %H:%M")}',
    '=' * 70,
    '',
    '--- PART 1: DIAGNOSIS ---',
    '',
    'Why was TF Signal OFF nearly empty (n=1 effective)?',
    'Signal OFF days lacked predicted_direction because OFF = transitional',
    'but NOT Signal ON, and Signal ON requires USE rating + direction mapping.',
    'Without predicted_direction, dir_hit returns NaN, leaving only 1 day.',
    '',
    'OOS transitional day breakdown (456 total):',
    '  ESTABLISHED (excluded by condition b):         31 days',
    '  EARLY/SETTLING, cell not USE (cond d fails):  193 days',
    '  EARLY/SETTLING, cell USE, no direction map:   182 days',
    '    (mostly F->F: USE-rated but no directional implication)',
    '  Signal ON (all conditions met):                50 days',
    '',
    'Calibration rating distribution (1w, transitional shapes):',
    '  MB (0.1): 2 USE (->SB, ->C), 3 non-USE (->MB, ->TB, ->F)',
    '  TB (0.2): 5 SKIP (all cells)',
    '  F  (2):   2 USE (->C, ->F), 3 non-USE (->SB, ->MB, ->TB)',
    '  Note: F->F is USE-rated but has no direction mapping',
    '        (F staying F has no directional trade implication)',
    '',
    '--- PART 2: SIGNAL OFF REDEFINITION ---',
    '',
    'Fix chosen: Option 3 (unconditional baseline)',
    'Signal OFF = ALL transitional days with TM prediction and direction',
    'mapping, EXCLUDING Signal ON days. Removes duration_stage and USE',
    'filters from the OFF definition, keeping them only for ON.',
    '',
    'This tests: does the USE + EARLY/SETTLING filter improve directional',
    'hit rate over unfiltered TM predictions on transitional days?',
    'Signal ON and OFF now both have predicted_direction -> comparable hit rates.',
    '',
    '--- PART 3: TF RESULTS (corrected) ---',
    '',
]

# Per-pair detail
for _, row in tf_fixed_df.iterrows():
    pair = row['pair']
    log_lines.append(f'  [{pair}]')
    for period in ['IS', 'OOS']:
        n_on = row[f'{period}_n_on']
        n_off = row[f'{period}_n_off']
        n_ep_on = row[f'{period}_n_episodes_on']
        n_ep_off = row[f'{period}_n_episodes_off']
        log_lines.append(f'    {period}: ON={n_on} ({n_ep_on} ep), OFF={n_off} ({n_ep_off} ep)')
        for h in [5, 10]:
            hit_on = row.get(f'{period}_on_hit_{h}d', '-')
            hit_off = row.get(f'{period}_off_hit_{h}d', '-')
            hit_on_n = row.get(f'{period}_on_hit_{h}d_n', 0)
            hit_off_n = row.get(f'{period}_off_hit_{h}d_n', 0)
            confirm = row.get(f'{period}_target_confirm_{h}d', '-')
            log_lines.append(f'      @{h}d: ON hit={hit_on}% (n={hit_on_n}) '
                           f'OFF hit={hit_off}% (n={hit_off_n}) '
                           f'target_confirm={confirm}%')
    for ag in ['agree', 'disagree']:
        n_ag = row.get(f'OOS_{ag}_n', 0)
        hit_ag = row.get(f'OOS_{ag}_hit_5d', '-')
        log_lines.append(f'    TM 1w/2w {ag}: n={n_ag} hit@5d={hit_ag}%')
    log_lines.append('')

# Before/after comparison
log_lines.append('--- BEFORE/AFTER COMPARISON (OOS, 5d) ---')
log_lines.append(f'  {"Pair":<10} {"OLD OFF n":>9} {"OLD OFF hit":>12} {"NEW OFF n":>9} {"NEW OFF hit":>12} {"ON hit":>8}')
log_lines.append(f'  {"-"*62}')

# Old results from strat_b_df
for i, (_, new_row) in enumerate(tf_fixed_df.iterrows()):
    pair = new_row['pair']
    old_row = strat_b_df[strat_b_df['pair'] == pair].iloc[0]
    old_n = old_row.get('OOS_off_hit_5d_n', 0)
    old_hit = old_row.get('OOS_off_hit_5d', np.nan)
    old_hit_s = f'{old_hit:.1f}%' if not np.isnan(old_hit) else '-'
    new_n = new_row.get('OOS_off_hit_5d_n', 0)
    new_hit = new_row.get('OOS_off_hit_5d', np.nan)
    new_hit_s = f'{new_hit:.1f}%' if not np.isnan(new_hit) else '-'
    on_hit = new_row.get('OOS_on_hit_5d', np.nan)
    on_hit_s = f'{on_hit:.1f}%' if not np.isnan(on_hit) else '-'
    log_lines.append(f'  {pair:<10} {old_n:>9} {old_hit_s:>12} {new_n:>9} {new_hit_s:>12} {on_hit_s:>8}')

# Cross-pair summary with gate check
log_lines.append('')
log_lines.append('--- TF CROSS-PAIR SUMMARY (corrected, OOS, 5d) ---')
log_lines.append(f'  {"Pair":<10} {"n ON":>6} {"n ep":>5} {"hit ON":>8} {"n OFF":>6} {"hit OFF":>8} {"delta":>7} {"Gate":>12}')
log_lines.append(f'  {"-"*66}')

tf_gates = []
for _, row in tf_fixed_df.iterrows():
    pair = row['pair']
    n_on = row['OOS_n_on']
    n_ep = row['OOS_n_episodes_on']
    n_off = row['OOS_n_off']
    hit_on = row.get('OOS_on_hit_5d', np.nan)
    hit_off = row.get('OOS_off_hit_5d', np.nan)

    if n_on < 10:
        gate = 'INSUFFICIENT'
    elif n_on < 30:
        if not np.isnan(hit_on) and not np.isnan(hit_off) and (hit_on - hit_off) > 5:
            gate = 'PASS*'
        else:
            gate = 'INSUFFICIENT'
    else:
        if not np.isnan(hit_on) and not np.isnan(hit_off) and (hit_on - hit_off) > 5:
            gate = 'PASS'
        else:
            gate = 'FAIL'

    delta = round(hit_on - hit_off, 1) if not np.isnan(hit_on) and not np.isnan(hit_off) else '-'
    hit_on_s = f'{hit_on:.1f}%' if not np.isnan(hit_on) else '-'
    hit_off_s = f'{hit_off:.1f}%' if not np.isnan(hit_off) else '-'

    log_lines.append(f'  {pair:<10} {n_on:>6} {n_ep:>5} {hit_on_s:>8} {n_off:>6} {hit_off_s:>8} {str(delta)+"pp":>7} {gate:>12}')
    tf_gates.append({'pair': pair, 'gate': gate, 'n_on': n_on, 'n_off': n_off, 'delta': delta})

# Gate verdict
log_lines.append('')
log_lines.append('--- GATE CHECK VERDICT (TF only, corrected) ---')
b_passes = [g for g in tf_gates if g['gate'] in ('PASS', 'PASS*')]
b_fails = [g for g in tf_gates if g['gate'] == 'FAIL']
b_insuf = [g for g in tf_gates if g['gate'] == 'INSUFFICIENT']
log_lines.append(f'TF: {len(b_passes)} PASS, {len(b_fails)} FAIL, {len(b_insuf)} INSUFFICIENT')
if b_passes:
    log_lines.append(f'  Passing pairs: {", ".join(g["pair"] for g in b_passes)}')
    log_lines.append(f'  -> These pairs proceed to Stage 1 (MR already has >=1 PASS*)')
else:
    log_lines.append(f'  -> NO passing pairs under corrected definition.')

log_lines.append('')
log_lines.append('MR (Strategy A) results unchanged from first run — not rerun.')

content = '\n'.join(log_lines)
print(content)

with open(LOG_FILE, 'a', encoding='utf-8') as f:
    f.write('\n' + content + '\n')
print(f'\n[Appended to {LOG_FILE}]')

Running TF (fixed Signal OFF)...
  M1-M2: ON=50 (19 ep), OFF=1 (1 ep), hit@5d ON=44.0% (n=50) OFF=100.0% (n=1)


  M2-M3: ON=50 (19 ep), OFF=1 (1 ep), hit@5d ON=48.0% (n=50) OFF=100.0% (n=1)


  M3-M4: ON=50 (19 ep), OFF=1 (1 ep), hit@5d ON=44.0% (n=50) OFF=0.0% (n=1)
  M4-M5: ON=50 (19 ep), OFF=1 (1 ep), hit@5d ON=40.0% (n=50) OFF=0.0% (n=1)


  M5-M6: ON=50 (19 ep), OFF=1 (1 ep), hit@5d ON=46.0% (n=50) OFF=0.0% (n=1)

STAGE 0 — TF SIGNAL OFF FIX & RERUN — 2026-07-10 13:38

--- PART 1: DIAGNOSIS ---

Why was TF Signal OFF nearly empty (n=1 effective)?
Signal OFF days lacked predicted_direction because OFF = transitional
but NOT Signal ON, and Signal ON requires USE rating + direction mapping.
Without predicted_direction, dir_hit returns NaN, leaving only 1 day.

OOS transitional day breakdown (456 total):
  ESTABLISHED (excluded by condition b):         31 days
  EARLY/SETTLING, cell not USE (cond d fails):  193 days
  EARLY/SETTLING, cell USE, no direction map:   182 days
    (mostly F->F: USE-rated but no directional implication)
  Signal ON (all conditions met):                50 days

Calibration rating distribution (1w, transitional shapes):
  MB (0.1): 2 USE (->SB, ->C), 3 non-USE (->MB, ->TB, ->F)
  TB (0.2): 5 SKIP (all cells)
  F  (2):   2 USE (->C, ->F), 3 non-USE (->SB, ->MB, ->TB)
  Note: F->F is USE-rated but ha